# 01 — Data Acquisition

This notebook downloads and caches all datasets needed for the Autobahn speed safety analysis.

**Datasets:**
1. 🇩🇪 **Unfallatlas** — GPS-level individual accident records (Germany, 2016–2023)
2. 🇩🇪 **Destatis GENESIS** — Aggregated accident stats by road type (Germany)
3. 🇳🇱 **CBS OData** — Road accident casualties by road type (Netherlands)
4. 🇳🇱 **BRON** — Individual accident records (Netherlands) — manual download
5. 🇪🇺 **ERSO/CARE** — EU comparative road safety statistics

Run cells in order. All data is saved to `../data/raw/` (gitignored).

---

**Prerequisites:**
- Copy `.env.example` to `.env` and fill in Destatis credentials (free registration at https://www-genesis.destatis.de/)
- `uv sync` to install dependencies

In [ ]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

from autobahn_safety.data_loaders import (
    download_unfallatlas,
    load_unfallatlas,
    fetch_destatis_genesis,
    fetch_cbs_odata,
)

load_dotenv()

DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_RAW.resolve()}")

## 1. Germany — Unfallatlas

Individual GPS-located accident records for all of Germany, 2016–2023.  
Published jointly by the Statistische Landesämter.

**Source:** https://unfallatlas.statistikportal.de/  
**Direct URL pattern:** `https://unfallatlas.statistikportal.de/_data/Unfallorte{YEAR}_EPSG25832_CSV.zip`

Key fields:
- `UKATEGORIE`: 1=fatal, 2=serious injury, 3=slight injury
- `INN_ORT`: 0 = outside built-up area (includes Autobahn)
- `XGCSWGS84`, `YGCSWGS84`: WGS84 coordinates

In [ ]:
UNFALLATLAS_DIR = DATA_RAW / "germany" / "unfallatlas"
YEARS = list(range(2016, 2024))  # 2016–2023

for year in YEARS:
    year_dir = UNFALLATLAS_DIR / str(year)
    existing = list(year_dir.glob("*.csv")) if year_dir.exists() else []
    if existing:
        print(f"{year}: already downloaded ({existing[0].name})")
        continue
    print(f"Downloading {year}...")
    try:
        download_unfallatlas(year, year_dir)
        print(f"{year}: done")
    except Exception as e:
        print(f"{year}: FAILED — {e}")

In [ ]:
# Load all years into a single DataFrame
df_unfallatlas = load_unfallatlas(UNFALLATLAS_DIR, YEARS)
print(f"Loaded {len(df_unfallatlas):,} accident records")
print(f"Columns: {list(df_unfallatlas.columns)}")
df_unfallatlas.head()

## 2. Germany — Destatis GENESIS-Online

Aggregated accident statistics by road type (Autobahn, Bundesstraße, etc.) and severity.  
Useful for long-term trends and normalization with official vehicle-km figures.

**Requires:** Free account at https://www-genesis.destatis.de/  
**Set in `.env`:** `DESTATIS_USERNAME` and `DESTATIS_PASSWORD`

**Key tables:**
- `46241-0023` — accidents by category and location type
- `46241-0031` — accidents on motorways (Autobahn) specifically

In [ ]:
DESTATIS_DIR = DATA_RAW / "germany" / "destatis"
DESTATIS_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "46241-0023": "accidents_by_road_type",
    "46241-0031": "motorway_accidents",
}

for table_id, name in tables.items():
    out_path = DESTATIS_DIR / f"{name}.csv"
    if out_path.exists():
        print(f"{table_id}: already downloaded")
        continue
    try:
        df = fetch_destatis_genesis(table_id, start_year=2000, end_year=2023)
        df.to_csv(out_path, index=False)
        print(f"{table_id}: {len(df)} rows saved to {out_path.name}")
    except ValueError as e:
        print(f"Skipped: {e}")
    except Exception as e:
        print(f"{table_id}: FAILED — {e}")

## 3. Netherlands — CBS OData API

Road accident casualty statistics by road type, available via CBS (Statistics Netherlands) open API.  
No registration required.

**Portal:** https://opendata.cbs.nl/  
**API base:** `https://opendata.cbs.nl/ODataApi/odata/{DATASET_ID}/`

Search CBS StatLine for:
- *verkeersslachtoffers* (traffic casualties)
- *verkeersongevallen* (traffic accidents)
- Filter by road type: *autosnelweg* = motorway

In [ ]:
CBS_DIR = DATA_RAW / "netherlands" / "cbs"
CBS_DIR.mkdir(parents=True, exist_ok=True)

# Discover dataset structure first
# Replace dataset_id with the correct one from CBS StatLine search
# Common candidates: "71738NED", "81395NED", "7418ENG"
dataset_id = "71738NED"  # UPDATE if this is not the right dataset

import requests

# First, check what tables this dataset has
meta_url = f"https://opendata.cbs.nl/ODataApi/odata/{dataset_id}/"
resp = requests.get(meta_url, timeout=30)
if resp.ok:
    print("Available tables:")
    for item in resp.json().get("value", []):
        print(f"  - {item['name']}: {item.get('url', '')}")
else:
    print(f"Failed to fetch metadata: {resp.status_code}")

In [ ]:
# Fetch the actual data
out_path = CBS_DIR / f"{dataset_id}.csv"

if out_path.exists():
    df_cbs = pd.read_csv(out_path)
    print(f"Loaded from cache: {len(df_cbs)} rows")
else:
    df_cbs = fetch_cbs_odata(dataset_id)
    df_cbs.to_csv(out_path, index=False)
    print(f"Fetched and saved: {len(df_cbs)} rows")

df_cbs.head()

## 4. Netherlands — BRON (Individual accident records)

BRON = *Bestand geRegistreerde Ongevallen in Nederland* — the official Dutch individual accident database.  
Maintained by CBS, SWOV, and Rijkswaterstaat.

**Download:**
1. Go to https://data.overheid.nl and search for **"BRON verkeersongevallen"**
2. Or request from SWOV: https://www.swov.nl/feiten-cijfers/bronnen-en-methoden/bron
3. Extract to `../data/raw/netherlands/bron/`

**Key fields:** road type, speed limit, severity, coordinates (RD New / WGS84), year

In [ ]:
BRON_DIR = DATA_RAW / "netherlands" / "bron"
BRON_DIR.mkdir(parents=True, exist_ok=True)

bron_files = list(BRON_DIR.glob("*.csv")) + list(BRON_DIR.glob("*.xlsx"))
if bron_files:
    print(f"Found BRON files: {[f.name for f in bron_files]}")
    # Load and inspect
    df_bron = pd.read_csv(bron_files[0]) if bron_files[0].suffix == ".csv" else pd.read_excel(bron_files[0])
    print(f"Shape: {df_bron.shape}")
    df_bron.head()
else:
    print("⚠️  No BRON data found.")
    print(f"   → Download manually and place in: {BRON_DIR.resolve()}")
    print("   → https://data.overheid.nl (search 'BRON verkeersongevallen')")

## 5. EU Comparative — ERSO / CARE Database

The European Road Safety Observatory provides standardized cross-country statistics.  
Useful for a clean DE vs NL comparison on a common baseline.

**Portal:** https://road-safety.transport.ec.europa.eu/  
**Country profiles:** DE and NL, statistics by road type, 1991–present

Manual download steps:
1. Go to the ERSO portal
2. Navigate to Statistics → Country profiles
3. Download Germany and Netherlands fatality tables
4. Save to `../data/raw/eu/erso/`

In [ ]:
ERSO_DIR = DATA_RAW / "eu" / "erso"
ERSO_DIR.mkdir(parents=True, exist_ok=True)

erso_files = list(ERSO_DIR.glob("*.csv")) + list(ERSO_DIR.glob("*.xlsx"))
if erso_files:
    print(f"Found ERSO files: {[f.name for f in erso_files]}")
else:
    print("⚠️  No ERSO data found.")
    print(f"   → Download from https://road-safety.transport.ec.europa.eu/")
    print(f"   → Place in: {ERSO_DIR.resolve()}")

## Summary

| Dataset | Status | Records | Notes |
|---------|--------|---------|-------|
| Unfallatlas (DE) | — | — | Run cells above |
| Destatis GENESIS (DE) | — | — | Needs .env credentials |
| CBS OData (NL) | — | — | Auto-fetched |
| BRON (NL) | — | — | Manual download |
| ERSO/CARE (EU) | — | — | Manual download |

→ Proceed to `02_data_exploration.ipynb` once data is downloaded.